In [1]:
!pip install -q \
transformers==4.52.4 \
huggingface_hub==0.33.4 \
accelerate==1.8.1 \
datasets==3.6.0 \
openai-whisper \
librosa \
soundfile \
sentencepiece

In [ ]:

!pip -q install -U transformers accelerate datasets
!pip -q install -U openai-whisper
!pip -q install -U librosa soundfile ffmpeg-python
!pip -q install -U jiwer

print("="*70)
print("Libraries Installed Successfully!")
print("="*70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 108.5 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
Libraries Installed Successfully!


In [3]:
import transformers
import huggingface_hub

print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

Transformers: 5.14.1
Hugging Face Hub: 1.24.0


In [ ]:

import os
import json
import warnings
import numpy as np
import pandas as pd
import librosa
import soundfile as sf

import torch
import whisper

from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification
)

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"

print("="*70)
print("="*70)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

print("Device  :", device)

print("="*70)

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4
Device  : cuda


In [ ]:

print("="*70)
print("Loading Whisper...")
print("="*70)

whisper_model = whisper.load_model("base")

print("✓ Whisper Loaded")

print("\n" + "="*70)
print("Loading Speech Emotion Recognition Model...")
print("="*70)

emotion_classifier = pipeline(
    task="audio-classification",
    model="Dpngtm/wav2vec2-emotion-recognition",
    device=0 if torch.cuda.is_available() else -1
)

print("✓ Speech Emotion Model Loaded")

print("="*70)

Loading Whisper...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 361MiB/s]


✓ Whisper Loaded

Loading Speech Emotion Recognition Model...


config.json:   0%|          | 0.00/2.37k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/257 [00:00<?, ?B/s]

✓ Speech Emotion Model Loaded


In [6]:
import os

print("Contents of /kaggle/input:")

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    for file in files:
        print("   ", file)

Contents of /kaggle/input:
/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/ngay03112993
/kaggle/input/datasets/ngay03112993/distilbert
    config.json
    tokenizer.json
    tokenizer_config.json
    model.safetensors
    label2id.json
    id2label.json
/kaggle/input/datasets/ngay03112993/voice-input
    WhatsApp Ptt 2026-07-18 at 10.41.26 PM.ogg


In [ ]:


audio_path = "/kaggle/input/datasets/ngay03112993/voice-input/WhatsApp Ptt 2026-07-18 at 10.41.26 PM.ogg"

print("Audio File:")
print(audio_path)

Audio File:
/kaggle/input/datasets/ngay03112993/voice-input/WhatsApp Ptt 2026-07-18 at 10.41.26 PM.ogg


In [ ]:

print("=" * 70)
print("Transcribing Audio...")
print("=" * 70)

result = whisper_model.transcribe(audio_path, fp16=False)

transcript = result["text"].strip()

print("\nTranscript:")
print("-" * 70)
print(transcript)
print("-" * 70)

Transcribing Audio...

Transcript:
----------------------------------------------------------------------
I had a friend very close to me but recently after like many things have happened so I feel like I again gave her mental pressure she used to be happy I made sure that she won't get to undergo all those stuff that she undergone in her bit it but again I heard her I feel really very bad I don't want to speak to anyone I want to be alone I want to be away from everyone's life I just want to end my life I don't know to say anything
----------------------------------------------------------------------


In [ ]:
print("=" * 70)
print("Predicting Speech Emotion...")
print("=" * 70)

speech_predictions = emotion_classifier(audio_path)

print("\nTop Speech Emotion Predictions:\n")

for pred in speech_predictions:
    print(f"{pred['label']:15s} : {pred['score']:.4f}")

Predicting Speech Emotion...

Top Speech Emotion Predictions:

sad             : 0.6478
fearful         : 0.3367
disgust         : 0.0076
calm            : 0.0051
happy           : 0.0016
surprised       : 0.0008
angry           : 0.0003


In [10]:
import os

print("Checking saved DistilBERT model...\n")

print("Model folder exists:",
      os.path.exists("/kaggle/working/final_distilbert_model"))

print("ZIP exists:",
      os.path.exists("/kaggle/working/final_distilbert_model.zip"))

Checking saved DistilBERT model...

Model folder exists: False
ZIP exists: False


In [ ]:
import os

print("="*70)
print("Contents of /kaggle/input")
print("="*70)

for root, dirs, files in os.walk("/kaggle/input"):
    print(f"\n📁 {root}")
    for d in dirs:
        print(f"   📂 {d}")
    for f in files:
        print(f"   📄 {f}")

Contents of /kaggle/input

📁 /kaggle/input
   📂 datasets

📁 /kaggle/input/datasets
   📂 ngay03112993

📁 /kaggle/input/datasets/ngay03112993
   📂 distilbert
   📂 voice-input

📁 /kaggle/input/datasets/ngay03112993/distilbert
   📄 config.json
   📄 tokenizer.json
   📄 tokenizer_config.json
   📄 model.safetensors
   📄 label2id.json
   📄 id2label.json

📁 /kaggle/input/datasets/ngay03112993/voice-input
   📄 WhatsApp Ptt 2026-07-18 at 10.41.26 PM.ogg


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Detect device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_PATH = "/kaggle/input/datasets/ngay03112993/distilbert"

print("="*70)
print("Loading DistilBERT...")
print("="*70)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

text_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

text_model.to(device)
text_model.eval()

print(f"Device : {device}")
print("✅ DistilBERT Loaded Successfully!")
print("="*70)

Loading DistilBERT...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Device : cuda
✅ DistilBERT Loaded Successfully!


In [15]:
# Read emotion names
import json

with open(f"{MODEL_PATH}/id2label.json", "r") as f:
    id2label = json.load(f)

print("="*70)
print("Predicted Text Emotions")
print("="*70)

predicted_emotions = []

for idx, prob in enumerate(probabilities):
    if prob >= threshold:
        emotion = id2label[str(idx)]
        predicted_emotions.append({
            "emotion": emotion,
            "confidence": float(prob)
        })

if len(predicted_emotions) == 0:
    print("No emotion exceeded the threshold.")
else:
    for item in sorted(predicted_emotions,
                       key=lambda x: x["confidence"],
                       reverse=True):
        print(f"{item['emotion']:15s} : {item['confidence']:.4f}")

Predicted Text Emotions
sadness         : 0.9958
disappointment  : 0.9475


In [ ]:
import json

results = {
    "transcript": transcript,
    "speech_emotion_predictions": speech_predictions,
    "text_emotion_predictions": predicted_emotions
}

output_file = "/kaggle/working/voice_analysis_results.json"

with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print("=" * 70)
print("Voice Analysis Completed Successfully!")
print("=" * 70)
print(f"Results saved to: {output_file}")

Voice Analysis Completed Successfully!
Results saved to: /kaggle/working/voice_analysis_results.json
